In [66]:
import os
import pickle
import numpy as np
import pandas as pd
from tqdm import tqdm
import gzip
import json
from collections import defaultdict

In [67]:
dataset = "fashion"

In [2]:
def parse(path): # for Amazon
    g = gzip.open(path, 'rb')
    inter_list = []
    for l in tqdm(g):
        inter_list.append(json.loads(l.decode()))
        # inter_list.append(eval(l))

    return inter_list

In [4]:
def Amazon(dataset_name, rating_score=0.0):
    datas = []
    # older Amazon
    data_flie = '/home/lgr/LLM-ESR/data/' + str(dataset_name) + '/raw/' + str(dataset_name) + '.json.gz'
    # latest Amazon
    # data_flie = '/home/hui_wang/data/new_Amazon/' + dataset_name + '.json.gz'
    for inter in parse(data_flie):
        if float(inter['overall']) <= rating_score: # 小于一定分数去掉
            continue
        user = inter['reviewerID']
        item = inter['asin']
        time = inter['unixReviewTime']
        datas.append((user, item, int(time)))
    return datas

In [8]:
def filter_common(user_items, user_t, item_t):
    """过滤掉交互次数小于user_t和item_t的用户和物品"""

    user_count = defaultdict(int)
    item_count = defaultdict(int)
    for user, item, _ in user_items:
        user_count[user] += 1
        item_count[item] += 1

    User = {}
    for user, item, timestamp in user_items:
        if user_count[user] < user_t or item_count[item] < item_t:
            continue
        if user not in User.keys():
            User[user] = []
        User[user].append((item, timestamp))
    # 此时 User中已经完成了过滤。但是还是有一些用户的交互次数小于3.这是因为虽然用户的交互物品长度本来大于3，但是其中物品有小于3的。所以删掉了这部分物品
    new_User = {}
    for userid in User.keys():
        User[userid].sort(key=lambda x: x[1])
        new_hist = [i for i, t in User[userid]]
        new_User[userid] = new_hist

    return new_User

In [40]:
def get_counts(user_items):

    user_count = {}
    item_count = {}

    for user, items in user_items.items():
        user_count[user] = len(items)
        for item in items:
            if item not in item_count.keys():
                item_count[item] = 1
            else:
                item_count[item] += 1

    return user_count, item_count

In [56]:
processed = Amazon(dataset)

883636it [00:14, 59057.42it/s]


In [57]:
filter_data = filter_common(processed,3,5)

In [59]:
user_count, item_count = get_counts(filter_data)

In [60]:
item_count

{'B00MW2FEWS': 1,
 'B00007GDFV': 1,
 'B00B6AWXYE': 2,
 'B00EZMCOCQ': 1,
 'B00IQZZ9JI': 4,
 'B00KA3UV0Q': 53,
 'B00KW4L9XQ': 57,
 'B00008JOQI': 2,
 'B0002Z1JNK': 15,
 'B00125PTY4': 29,
 'B00513PYSO': 3,
 'B012ZQR84C': 2,
 'B00062NHH0': 4,
 'B005N7YWX6': 65,
 'B01535OKZM': 1,
 'B01D35B1TW': 1,
 'B002EX434U': 11,
 'B00DURK0E6': 11,
 'B00063VWTY': 1,
 'B00066G516': 12,
 'B00GQ4MMXW': 12,
 'B00B77I7T6': 2,
 'B00WHUW5KY': 4,
 'B00BWA0P7K': 7,
 'B0083QBJ1W': 3,
 'B00XNCUNVS': 4,
 'B0105EFX1Q': 10,
 'B0111KTNM8': 2,
 'B00JC1RTOS': 8,
 'B00JXH7ZTK': 1,
 'B00SYB9OR2': 1,
 'B01FI5PDGC': 2,
 'B00TZD3AI2': 3,
 'B005JDA5JY': 1,
 'B0014F8TIU': 228,
 'B000673JT6': 2,
 'B001XOQTSE': 15,
 'B014FH7AKQ': 3,
 'B0006HB4XE': 8,
 'B011HVE6PU': 6,
 'B001251DPI': 1,
 'B00MLYE8PQ': 29,
 'B000GWQKW4': 1,
 'B00VAVSB6S': 1,
 'B00GN7XZNS': 9,
 'B00SI020DE': 6,
 'B011NU8L7E': 1,
 'B00KVMMIKC': 1,
 'B0006SUTRK': 1,
 'B0002X9A5G': 1,
 'B0007MV6PO': 2,
 'B000YFSR5G': 1953,
 'B001LNSY2Q': 25,
 'B00G9GYVZ4': 1,
 'B00JE87E

In [38]:
min(item_count.values())

1

In [64]:
inter = {}

with open(f"./{dataset}/handled/inter.txt", 'r') as f:
    for line in tqdm(f):
        line_data = line.rstrip().split(' ')
        user_id = line_data[0]
        line_data.pop(0)
        if user_id in inter:
            inter[user_id].append(line_data[0])
        else:
            inter[user_id] = [line_data[0]]

34759it [00:00, 424567.87it/s]


In [27]:
inter

{'1': ['1', '2', '3'],
 '2': ['4', '5', '5'],
 '3': ['4', '5', '5'],
 '4': ['4', '5', '5', '6'],
 '5': ['4', '5', '5'],
 '6': ['4', '5', '5'],
 '7': ['4', '5', '5'],
 '8': ['4', '5', '5'],
 '9': ['4', '5', '5'],
 '10': ['4', '5', '5'],
 '11': ['4', '5', '5'],
 '12': ['4', '5', '5'],
 '13': ['4', '5', '5'],
 '14': ['4', '5', '5'],
 '15': ['4', '5', '5'],
 '16': ['7', '8', '9'],
 '17': ['7', '10', '10'],
 '18': ['11', '12', '13'],
 '19': ['14', '15', '15', '16'],
 '20': ['14', '17', '18'],
 '21': ['19', '19', '20'],
 '22': ['21', '22', '20'],
 '23': ['23', '24', '25', '26'],
 '24': ['25', '27', '27', '27', '27'],
 '25': ['28', '29', '30', '31'],
 '26': ['32', '33', '34'],
 '27': ['35', '36', '37'],
 '28': ['38', '36', '37'],
 '29': ['39', '36', '37'],
 '30': ['40', '36', '37'],
 '31': ['41', '42', '36', '37'],
 '32': ['43', '36', '37'],
 '33': ['36', '37', '44'],
 '34': ['45', '36', '37'],
 '35': ['46', '36', '37'],
 '36': ['47', '36', '37'],
 '37': ['36', '37', '48'],
 '38': ['36', '37'

In [74]:
inter_seq = {}

with open(f"./{dataset}/handled/inter_seq.txt", 'r') as f:
    for line in tqdm(f):
        line_data = line.rstrip().split(' ')
        user_id = line_data[0]
        line_data.pop(0)    # delete user_id
        inter_seq[user_id] = line_data

22379it [00:00, 525732.77it/s]


In [65]:
user_count, item_count = get_counts(inter)
user_count_list = list(user_count.values())
item_count_list = list(item_count.values())
user_avg, user_min, user_max = np.mean(user_count_list), np.min(user_count_list), np.max(user_count_list)
item_avg, item_min, item_max = np.mean(item_count_list), np.min(item_count_list), np.max(item_count_list)
interact_num = np.sum([x for x in user_count_list])
sparsity = (1 - interact_num / (len(user_count) * len(item_count))) * 100
show_info = f'Total User: {len(user_count)}, Avg User: {user_avg:.4f}, Min Len: {user_min}, Max Len: {user_max}\n' + \
            f'Total Item: {len(item_count)}, Avg Item: {item_avg:.4f}, Min Inter: {item_min}, Max Inter: {item_max}\n' +\
            f'Iteraction Num: {interact_num}, Sparsity: {sparsity:.2f}%'
print(show_info)

Total User: 9094, Avg User: 3.8222, Min Len: 3, Max Len: 26
Total Item: 4722, Avg Item: 7.3611, Min Inter: 2, Max Inter: 1914
Iteraction Num: 34759, Sparsity: 99.92%


In [75]:
user_count, item_count = get_counts(inter_seq)
user_count_list = list(user_count.values())
item_count_list = list(item_count.values())
user_avg, user_min, user_max = np.mean(user_count_list), np.min(user_count_list), np.max(user_count_list)
item_avg, item_min, item_max = np.mean(item_count_list), np.min(item_count_list), np.max(item_count_list)
interact_num = np.sum([x for x in user_count_list])
sparsity = (1 - interact_num / (len(user_count) * len(item_count))) * 100
show_info = f'Total User: {len(user_count)}, Avg User: {user_avg:.4f}, Min Len: {user_min}, Max Len: {user_max}\n' + \
            f'Total Item: {len(item_count)}, Avg Item: {item_avg:.4f}, Min Inter: {item_min}, Max Inter: {item_max}\n' +\
            f'Iteraction Num: {interact_num}, Sparsity: {sparsity:.2f}%'
print(show_info)

Total User: 22379, Avg User: 3.8093, Min Len: 3, Max Len: 40
Total Item: 36314, Avg Item: 2.3475, Min Inter: 1, Max Inter: 1953
Iteraction Num: 85248, Sparsity: 99.99%


In [21]:
num = 0
user_iter_num = {}
for user_id in data:
    user_iter_num[user_id] = len(data[user_id])
    num += len(data[user_id])
print(f"样本数为：{num}")

样本数为：394908


In [31]:
len(data)

52204

In [28]:
data["52200"]

['57286', '57287', '57285']

In [ ]:
with open(f"./{dataset}/handled/inter.txt", 'w') as f:
    for user, item_list in tqdm(data.items()):
        for item in item_list:
            u = int(user)
            i = int(item)
            f.write('%s %s\n' % (u, i))

In [1]:
def process_inter_file(input_path, output_path):
    try:
        with open(input_path, 'r') as infile, open(output_path, 'w') as outfile:
            for line in infile:
                # 去除首尾空白字符并分割成两部分
                parts = line.strip().split()
                if len(parts) == 2:
                    # 转换为整数并加1
                    uid = int(parts[0]) + 1
                    iid = int(parts[1]) + 1
                    # 写入新文件
                    outfile.write(f"{uid} {iid}\n")
        print(f"处理完成，结果已保存到 {output_path}")
    except FileNotFoundError:
        print(f"错误：找不到文件 {input_path}")
    except Exception as e:
        print(f"处理时发生错误：{str(e)}")

# 使用示例
if __name__ == "__main__":
    input_file = "/home/lgr/LLM-ESR/data/steam/handled/inter_origin.txt"       # 输入文件路径
    output_file = "/home/lgr/LLM-ESR/data/steam/handled/inter.txt" # 输出文件路径
    process_inter_file(input_file, output_file)

处理完成，结果已保存到 /home/lgr/LLM-ESR/data/steam/handled/inter.txt
